# CLIP Sanity Check: Practical Internal Architecture
**Project:** AI Video Investigator (WP3)
**Objective:** Demonstrate the mechanics of the CLIP model—specifically how it maps images and text into a shared **Joint Latent Space**.

--- 

### 1. How CLIP Works: The "Under the Hood" Explanation

#### A. Text Tokenization (Text Encoder)
- **Mechanism:** CLIP uses Byte-Pair Encoding (BPE). It breaks the sentence "a person jumping over a fence" into discrete tokens (e.g., `[49406, 320, 1541, ... 49407]`).
- **Contextualization:** These tokens pass through a Transformer. The final hidden state of the `[EOS]` (End of Sentence) token is typically used as the "global summary" of the text.
- **Projection:** This summary is projected via a linear layer into a specific dimension (e.g., 512 or 768), creating the **Text Embedding**.

#### B. Image Patching (Vision Transformer - ViT)
- **Mechanism:** The image is not processed as a whole pixel grid. It is divided into fixed-size **patches** (e.g., 32x32 pixels).
- **Linear Projection:** Each patch is flattened and projected into a vector. These "patch tokens" are then processed by a Transformer, similar to words in a sentence.
- **Global Feature:** A special `[CLS]` (Classification) token attends to all patches and aggregates the visual information into a single **Image Embedding**.

#### C. The Joint Latent Space & Cosine Similarity
- Both the Image and Text encoders are trained to output vectors of the **exact same length** (e.g., 512 floats).
- **Cosine Similarity** measures the angle between these two vectors. If the vectors point in the same direction, the similarity is high (~1.0), indicating the text accurately describes the image.

In [1]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F

# 1. Load the Model and Processor
# 'processor' handles both text tokenization and image resizing/normalization
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)

# 2. Load the Dummy Data
image = Image.open("sample_frame_2.png")
text_query = "a person jumping over a fence"

print(f"Processing Query: '{text_query}'")

# 3. Preprocessing: Preparing the payload for the model
# This converts the PIL image to a tensor and the text to token IDs
inputs = processor(text=[text_query], images=image, return_tensors="pt", padding=True)

with torch.no_grad():
    # 4. Extract Embeddings
    # Passing inputs directly to the model returns a structured output containing projected embeds
    outputs = model(**inputs)
    text_features = outputs.text_embeds
    image_features = outputs.image_embeds

    # 5. Dimensional Proof
    print("\n--- Latent Space Verification ---")
    print(f"Text Embedding Shape:  {text_features.shape}  (Batch Size, Latent Dim)")
    print(f"Image Embedding Shape: {image_features.shape} (Batch Size, Latent Dim)")
    
    if text_features.shape[1] == image_features.shape[1]:
        print(f"SUCCESS: Both modalities mapped to a {text_features.shape[1]}-dimensional space.")

    # 6. Calculate Cosine Similarity
    # We normalize the vectors first to calculate the cosine of the angle between them
    text_features = F.normalize(text_features, p=2, dim=-1)
    image_features = F.normalize(image_features, p=2, dim=-1)
    
    # Dot product of normalized vectors = Cosine Similarity
    similarity = (text_features @ image_features.T).item()

    print("\n--- Final Result ---")
    print(f"Cosine Similarity Match Score: {similarity:.4f}")
    print("(Score range: -1 to 1. Closer to 1 means high semantic alignment)")

C:\Ai_Expert\AI Video Investigator\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 19228.85it/s]


Processing Query: 'a person jumping over a fence'

--- Latent Space Verification ---
Text Embedding Shape:  torch.Size([1, 512])  (Batch Size, Latent Dim)
Image Embedding Shape: torch.Size([1, 512]) (Batch Size, Latent Dim)
SUCCESS: Both modalities mapped to a 512-dimensional space.

--- Final Result ---
Cosine Similarity Match Score: 0.1907
(Score range: -1 to 1. Closer to 1 means high semantic alignment)
